# HUDM 5001 — Programming for Data Science
## Session 06 · More pandas & SQLite Databases
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Week_06/06_Pandas_SQLite.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.  
>
>
> ▶️ **Please disable the generative AI feature first** —   
> If you are using Colab, follow Edit --> Notebook settings --> AI Assistance, check the box `Hide generative AI features`  
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.  
   
About this notebook:  
This notebook is adapted from Prof. Youmi Suk's materials for HUDM 5001, Programming for Data Science ([original repository](https://github.com/youmilab/hudm5001)).  
Wording in this notebook was refined with Claude. I am responsible for its accuracy.  

Version: Oct.12.2026 10:12 pm  

Two halves today, and they rhyme more than you would expect. First we go **deeper into pandas**: describing data honestly, cleaning up the mess, and stitching two tables together. Then we meet **SQLite**, a real database that lives in a single file — and you will notice that a `SELECT ... WHERE ... GROUP BY` is the same thought you already express with masks and `groupby`, just in another accent.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Produce a proper **descriptive summary** of a dataset, and say when the **median** beats the mean.
- Tidy a table: **rename**, **drop**, retype, and remove **duplicate** rows.
- Deal with **missing data** deliberately, rather than letting it quietly wreck your numbers.
- Summarize by group with **`groupby` + `agg`** and reshape with **`pivot_table`**.
- Combine tables with **`pd.concat`** (stacking) and **`pd.merge`** (joining on a key).
- Create a **SQLite** database, write tables into it, and query it with **SQL**.
- Read query results straight into a DataFrame with **`pd.read_sql_query`**, and translate between **SQL and pandas**.

### ✅ Prerequisites
- Sessions 01–05. Session 05's `groupby`, boolean masks, and `.loc`/`.iloc` do a lot of work today.
- Nothing to install: `sqlite3` ships with Python.

### 🧩 Key Concepts
`describe`  `agg`  `median vs mean`  `rename`  `drop`  `duplicates`  `missing data`  `pivot_table`  `concat`  `merge`  `SQLite`  `SELECT`  `WHERE`  `GROUP BY`  `JOIN`

### 📖 Reading
- McKinney, *Python for Data Analysis*, Chapters 7–8 (data cleaning, joining & reshaping)

### 🗺️ Today's Agenda
**Part 1 — More pandas**
1. Meet the data: the Titanic passenger list
2. Descriptive statistics that tell the truth
3. Tidying up: rename, drop, retype, de-duplicate
4. Missing data, handled on purpose
5. `groupby` + `agg`, and `pivot_table`
6. Combining tables: `concat` and `merge`

**Part 2 — SQLite**
7. Why a database?  
8. Create a database, a table, and some rows  
9. Querying with SQL: `SELECT`, `WHERE`, `ORDER BY`  
10. Aggregating with `GROUP BY`  
11. `JOIN` — the SQL cousin of `merge`  
12. pandas ↔ SQL, side by side

---
## ▶️ Setup — run this cell first
This imports what we need and points at two small tables in the course repo: the **Titanic passenger list** and a tiny **ports** lookup table we will join to it later.

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import sqlite3          # a full database engine, built into Python — nothing to install
import json, os

# Course data, read straight from the public repo (no downloads needed).
TITANIC_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/titanic.csv"
PORTS_URL   = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/ports.csv"

print("pandas", pd.__version__, "| sqlite3", sqlite3.sqlite_version)
print("Setup complete ✅")

---
# Part 1 — More pandas

## 2.1 Meet the data — the Titanic passenger list
891 passengers, one row each: whether they survived, their ticket class, sex, age, fare, and the port where they boarded. It is small enough to read and messy enough to be honest — there are real gaps in it, which is exactly what we want to practice on.

In [ ]:
titanic = pd.read_csv(TITANIC_URL)
# shape

# column

# descriptive info

# head


A quick note on the columns: `survived` is `1`/`0`, `pclass` is the ticket class (1st, 2nd, 3rd), `sibsp` counts siblings and spouses aboard, `parch` counts parents and children, and `embarked` is a port code (`S`, `C`, `Q`).

## 2.2 Descriptive statistics that tell the truth
`.describe()` shows you a whole numeric table at a glance. But the interesting part is what it *reveals*: look at the `fare` column below, where the mean and the 50% row (the median) are far apart.

In [ ]:
titanic[["age", "fare", "sibsp", "parch"]].describe().round(2)

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/skew.png" width="800">
</p>

Figure source: [medium.com](https://medium.com/geekculture/what-are-right-skewed-and-left-skewed-distributions-a29b3def7598)

The mean fare is about **32**, but the median is about **14** — and the max is over **512**. A handful of very expensive tickets drag the average upward. This is a **skewed** distribution, and it is the classic case where the **median** describes a typical passenger better than the mean does. Reporting only the mean here would quietly mislead your reader.

You can also ask for exactly the statistics you want with **`.agg()`**, passing a list of their names. The `agg` means `aggregate`, which applies one or more summary caculations across DataFrame rows, columns, or grouped subsets to condense large datasets into summary metrics.

In [ ]:
# pick your own statistics instead of taking describe() as-is
print(titanic["fare"].agg(["count", "mean", "median", "std", "min", "max"]).round(2))

# and the same for several columns at once
print(titanic[["age", "fare"]].agg(["mean", "median", "std"]).round(2))

Two more everyday summaries: **`value_counts()`** for categories (add `normalize=True` to get proportions instead of counts), and **`.corr()`** for how numeric columns move together.

In [ ]:
# how many passengers in each ticket class, as counts and as shares
print(titanic["pclass"].value_counts())
print(titanic["pclass"].value_counts(normalize=True).round(3))

# other methods to extract the columns

**Correlation** measures how two numeric columns move together, on a scale from **-1 to +1**. A value near `+1` means they rise together, near `-1` means one rises as the other falls, and near `0` means no straight-line relationship. Handing `.corr()` a DataFrame gives a whole grid — every column against every other — so the diagonal is always `1.00` (a column is perfectly correlated with itself).

In [ ]:
# the whole grid: every numeric column against every other
# this returns a dataframe
print(titanic[["age", "fare", "pclass", "survived"]].corr().round(2))


Read the `fare`/`pclass` cell: about **-0.55**, a solid negative relationship. That makes sense — `pclass` 1 is the *most* expensive class, so as the class number goes up, the fare goes down.

If you only care about **one pair**, call `.corr()` on a single column and pass the other. That returns one number instead of a grid, which is usually what you want in a sentence.

In [ ]:
# just one pair -> a single number
print("fare vs pclass:", round(titanic["fare"].corr(titanic["pclass"]), 3))
print("age vs fare   :", round(titanic["age"].corr(titanic["fare"]), 3))

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★★☆☆☆  **Skills:** `.mean()`, `.median()`, `.agg()`

For the `age` column: print its **mean** and its **median**, then print `count`, `min`, and `max` in one call with `.agg()`. Are the mean and median close together here? What does that tell you about the shape of the age distribution compared with `fare`?

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
print("mean  :", round(titanic["age"].mean(), 2))
print("median:", titanic["age"].median())
print(titanic["age"].agg(["count", "min", "max"]))
# mean ~29.7 and median 28.0 sit close together, so age is only mildly skewed --
# unlike fare, where the mean is more than twice the median.
```

</details>

## 2.3 Tidying up: rename, drop, retype, de-duplicate
Real tables arrive with awkward names, columns you do not need, and the occasional repeated row. Four tools handle nearly all of it. Notice that each one **returns a new table** rather than editing in place, so we assign the result to a variable.

In [ ]:
# work on a copy so the original stays intact
tidy = titanic.copy()

# 1. rename: give the cryptic columns readable names:
# the syntax: df = df.rename(collumns = {"old_col": "new_col"})
tidy = tidy.rename(columns={"sibsp": "n_siblings_spouses",
                            "parch": "n_parents_children"})
print("renamed:", list(tidy.columns))

# 2. drop:
# remove a column you do not need (axis=1 means columns)
# syntax: df = df.drop(columns=["col_1", "col_2"])
#         df = df.drop(["col_1", "col_2"], axis = 1)

# remove rows you do not need
# syntax: df = df.drop(index = [0, 1, 2])
#         df = df.drop([0,1,2], axis = 0)

slim = tidy.drop(columns=["n_siblings_spouses", "n_parents_children"])
print("after drop:", list(slim.columns))

# drop rows by condition:
# syntax: df = df.drop[df[df["col"] < 3.0].index]

In [ ]:
# 3. convert the type: survived is really a category, not a number to average
print("before:", titanic["survived"].dtype)
as_text = titanic["survived"].astype(str)   # <--- can be str, int, float
print("after :", as_text.dtype)

# 4. duplicates: how many exact repeat rows are there?
# syntax: dup_count = df.duplicated().sum()
#         df = df.drop_duplicates()

print("duplicate rows:", titanic.duplicated().sum())
no_dupes = titanic.drop_duplicates()
print("rows before:", titanic.shape[0], "| after:", no_dupes.shape[0])

For categorical variables (more than two categories), a computer does not understand the categories in text. It can read the numbers only. We need to convert the categorical variables into one-hot (dummy) encodings first.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/dummy_coding.png" width="800">
</p>

Figure source: [towardsdatascience.com](https://towardsdatascience.com/encoding-categorical-variables-one-hot-vs-dummy-encoding-6d5b9c46e2db/)

In [ ]:
# 4. convert a categorical variable into dummy coding
# syntax: df = pd.get_dummies(df, columns = ["col_cat"], drop_first=False, dtype=int)
tidy = pd.get_dummies(tidy, columns=["embarked"], drop_first=False, dtype=int)

This particular passenger list has no exact duplicates, which is good news — but `duplicated().sum()` is worth running on any dataset before you trust a count.

## 2.4 Missing data, handled on purpose
You met `NaN` briefly last week. Now let us take it seriously, because how you handle gaps changes your answers. Start by finding out where they are.

In [ ]:
# how many missing values in each column?
# df.isna(), df.dropna(), df.fillna()
print(titanic.isna().sum())

So `age` is missing for 177 passengers and `embarked` for 2. You have three defensible options, and the right one depends on your question:

1. **Drop the rows** — clean, but you throw away 177 real passengers.
2. **Fill the gaps** with a sensible value, usually the **median** for a skewed column.
3. **Leave them** — pandas already skips `NaN` when computing means, so sometimes doing nothing is fine.

Watch what each choice does to the row count and the average age.

In [ ]:
# option 1: drop rows missing an age
dropped = titanic.dropna(subset=["age"])
print("drop  -> rows:", dropped.shape[0], "| mean age:", round(dropped["age"].mean(), 2))

# option 2: fill missing ages with the median age
filled = titanic.copy()
filled["age"] = filled["age"].fillna(filled["age"].median())
print("fill  -> rows:", filled.shape[0], "| mean age:", round(filled["age"].mean(), 2))

# option 3: leave it alone -- mean() ignores NaN anyway
print("leave -> rows:", titanic.shape[0], "| mean age:", round(titanic["age"].mean(), 2))

Notice the mean barely moves but the **row count** changes a lot. Filling with the median keeps all 891 passengers and nudges the average only slightly — but it also invents data, so you should say so in your write-up. There is no free lunch here; there is only being explicit about what you did.

#### ✏️ Try It Yourself — Exercise 2
**Difficulty:** ★★★☆☆  **Skills:** `.isna()`, `.dropna()`, `.fillna()`, `.shape`

Work on a copy of `titanic`. (a) Print how many values are missing in `embarked`. (b) Drop the rows where `embarked` is missing and print the new row count. (c) On a *separate* copy, fill the missing `embarked` values with `"S"` (the most common port) and confirm with `.isna().sum()` that no gaps remain.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print("missing embarked:", titanic["embarked"].isna().sum())

no_missing = titanic.dropna(subset=["embarked"])
print("rows after dropping:", no_missing.shape[0])

patched = titanic.copy()
patched["embarked"] = patched["embarked"].fillna("S")
print("missing after filling:", patched["embarked"].isna().sum())
```

</details>

## 2.5 `groupby` + `agg`, and `pivot_table`
Last week `groupby` gave us one statistic per group. Pair it with **`.agg()`** and you get several at once — which is how a real descriptive table gets built.

In [ ]:
# survival rate by class: the mean of a 0/1 column IS the proportion who survived
print(titanic.groupby("pclass")["survived"].mean().round(3))

In [ ]:
# several statistics per group, in one table
print(titanic.groupby("pclass")["fare"].agg(["count", "mean", "median", "std"]).round(2))

A group summary is still a Series, so you can hand it straight back to plain Python with **`.to_dict()`** — handy when you want to drop the result into an f-string or save it as JSON.

In [ ]:
# a group summary, converted back into a Session-01 dictionary
fare_by_class = titanic.groupby("pclass")["fare"].mean().round(2).to_dict()
print(fare_by_class)
print("first class paid on average:", fare_by_class[1])

You can also group by **two** keys. Pass a list, and you get a row per combination.

In [ ]:
# survival rate for every class-and-sex combination
print(titanic.groupby(["pclass", "sex"])["survived"].mean().round(3))

That result is correct but a little cramped to read. **`pivot_table`** takes the same numbers and lays them out as a grid — one variable down the side, the other across the top. Same information, far easier on the eye.

In [ ]:
grid = pd.pivot_table(titanic,
                      values="survived",
                      index="pclass",      # rows
                      columns="sex",       # columns
                      aggfunc="mean").round(3)
print(grid)

Read that grid for a moment — it is the whole story of the disaster in six numbers. A woman in first class had roughly a 97% survival rate; a man in third class, about 14%.

> ⚠️ **One catch worth knowing:** `groupby` and `pivot_table` quietly **drop rows whose grouping key is missing**. Nobody warns you — the rows just are not there. If a grouping column has `NaN` values, check that the group totals still add up to the row count you started with.

In [ ]:
# grouping on a column with gaps loses rows -- embarked has 2 missing values
grouped_total = titanic.groupby("embarked")["survived"].count().sum()
print("rows in the table :", titanic.shape[0])
print("rows in the groups:", grouped_total, " <- the 2 passengers with no port fell out")

#### ✏️ Try It Yourself — Exercise 3
**Difficulty:** ★★★★☆  **Skills:** `groupby`, `.agg()`, `pivot_table`

(a) Group by `sex` and report `count`, `mean`, and `median` of `age` in one `.agg()` call. (b) Build a `pivot_table` of the **average `fare`**, with `pclass` down the side and `embarked` across the top, rounded to 2 decimals.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print(titanic.groupby("sex")["age"].agg(["count", "mean", "median"]).round(2))

fare_grid = pd.pivot_table(titanic,
                           values="fare",
                           index="pclass",
                           columns="embarked",
                           aggfunc="mean").round(2)
print(fare_grid)
```

</details>

## 2.6 Combining tables: `concat` and `merge`
Two different jobs, and mixing them up is a common beginner slip.  

**`pd.concat`** to stack/glue tables either by rows or by columns. The `concat` stands for `concatenate`, meaning `to connect or link`.  
**`pd.merge`** *joins* tables that share a **key column**.

### pd.concat()  
  
Suppose, we collect students information from two schools.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/concat_byrows.png" width="500">
</p>

Figure source: [swdevnotes.com/](https://swdevnotes.com/python/2020/pandas-concat-dataframes/)

In [ ]:
# School 1
students_1 = pd.DataFrame({'stu_id': [101, 102], 'sch_id': [1, 1]})

# School 2
students_2 = pd.DataFrame({'stu_id': [103, 104,105], 'sch_id': [2, 2, 2]})

In [ ]:
# to glue the students_1 and students_2 vertically
students_12 = pd.concat([students_1, students_2], axis=0, ignore_index=True)
print(students_12)

We can glue these two tables vertically. Note, to avoid confusion, please read `axis = 0` as `by rows` or `veritically`. Please set the `ignore_index=True` to avoid future merging issues.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/concat_bycols.png" width="500">
</p>

Figure source: [scaler.com/](https://www.scaler.com/topics/pandas/concatenate-two-dataframes-pandas/)

Now, suppose we have a set of GPAs for student 101 to student 105 as:

In [ ]:
stu_info = pd.DataFrame({'GPA': [3.1, 2.5, 3.2, 3.7, 2.9],
                         'Sex': [0, 1, 1, 0, 0]})      # 1 for male, 0 for female
print(stu_info)

We can glue these two tables horizontally. Note, to avoid confusion, please treat `axis = 1` as `horizontally`.

In [ ]:
students_12 = pd.concat([students_12, stu_info], axis=1)
print(students_12)

### pd.merge()

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/pd_merge.png" width="800">
</p>

Figure source: [practicaldatascience.org](https://practicaldatascience.org/notebooks/class_3/week_4/15.1_combining_merging_continued.html)

Supppose, we now have school information as:

In [ ]:
schools = pd.DataFrame({'sch_id':[1, 2],
                        'sch_name': ['Green High', 'Blue High'],
                        'location': ['North', 'South']})

print(schools)

We want to merge `schools` information into the `students_12` by using the `sch_id` as the key for paring.

In [ ]:
stu_sch = pd.merge(students_12, schools, how='left', on='sch_id')
print(stu_sch)

In [ ]:
# concat: stack the first 3 and the last 3 passengers into one 6-row table
top = titanic.head(3)
bottom = titanic.tail(3)
stacked = pd.concat([top, bottom])
print("stacked shape:", stacked.shape)
print(stacked[["passenger_id", "sex", "age"]])

Now the interesting one. Our `embarked` column holds bare codes — `S`, `C`, `Q` — which nobody wants to read. A second little table maps each code to a real port name, and `merge` glues them together on that shared column.

In [ ]:
ports = pd.read_csv(PORTS_URL)
print(ports)

In [ ]:
# join the passenger list to the port names on the shared "embarked" column
joined = pd.merge(titanic, ports, on="embarked", how="left")
print("shape before:", titanic.shape, "-> after:", joined.shape)
print(joined[["passenger_id", "embarked", "port_name", "country"]].head())

The `how=` argument decides what happens to rows that do not find a match:

| `how=` | keeps |
|---|---|
| `"left"` | every row of the **left** table (unmatched ones get `NaN`) |
| `"right"` | every row of the **right** table |
| `"inner"` | only rows that **match in both** |
| `"outer"` | **everything** from both sides |

Remember those two passengers with a missing `embarked`? With `how="left"` they survive the join but get a `NaN` port name. With `how="inner"` they quietly vanish — and the row count tells on us.

In [ ]:
inner = pd.merge(titanic, ports, on="embarked", how="inner")
print("left join rows :", joined.shape[0])
print("inner join rows:", inner.shape[0], "  <- the 2 passengers with no port are gone")
print("missing port names after the left join:", joined["port_name"].isna().sum())

> ⚠️ **Check the row count after every join.** If it changed and you did not expect it to, something is wrong with your key. That one habit will save you hours.

#### ✏️ Try It Yourself — Exercise 4
**Difficulty:** ★★★☆☆  **Skills:** `value_counts`, `groupby`, `.mean()`

Using the `joined` table: (a) count how many passengers boarded at each **`port_name`**; (b) compute the **average fare** for each `port_name` with `groupby`, rounded to 2 decimals.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
print(joined["port_name"].value_counts())
print(joined.groupby("port_name")["fare"].mean().round(2))
```

</details>

---
# Part 2 — SQLite databases

## 2.7 Why a database?
A CSV is fine until it is not. Databases earn their keep when data gets **big** (bigger than memory), when several people need it **at once**, when the data must **survive** your script, or when it lives in **many related tables** that need to stay consistent.

**SQLite** is the gentlest possible introduction: the entire database is a single file on disk, there is no server to run, and `sqlite3` is already part of Python. It is also the most deployed database on earth — it is inside your phone, your browser, and probably the app you used this morning.

The language for talking to it is **SQL**. Here is the good news: you already think in SQL. `SELECT` is choosing columns, `WHERE` is a boolean mask, `GROUP BY` is `groupby`. Same ideas, new accent.

### How big is big? From 1 MB to 1 PB

<div role="group" aria-label="Data-size comparison using the binary classroom convention: MB, GB, TB, PB. Each unit is 1,024 of the previous unit." style="max-width: 900px; padding: 20px 24px; background: #111827; color: #f1f5f9; border: 1px solid #334155; border-radius: 10px; font-family: system-ui, sans-serif;">
  <div style="font-size: 18px; font-weight: 700; margin-bottom: 6px;">Every step is 1,024× more data</div>
  <div style="font-size: 14px; color: #cbd5e1;">Start at 1 MB, then compare each larger unit with the unit immediately before it.</div>
  <div style="margin: 20px 0;">
    <div style="display: flex; flex-wrap: wrap; align-items: baseline; justify-content: space-between; gap: 4px 16px; margin-bottom: 8px;">
      <strong style="color: #38bdf8; font-size: 18px;">1 MB</strong>
      <span style="color: #e2e8f0; font-size: 15px;">Starting unit</span>
    </div>
    <div style="height: 18px; background: #243247; border-radius: 4px; overflow: hidden;">
      <div style="width: 25%; height: 100%; background: #38bdf8; border-radius: 4px;"></div>
    </div>
    <p style="margin: 9px 0 0; color: #cbd5e1; font-size: 14px; line-height: 1.65;"><strong>Titanic dataset:</strong> <a href="https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/titanic.csv" style="color: #7dd3fc; text-decoration: underline;" target="_blank" rel="noopener">Course Titanic CSV</a> is only <strong>0.026 MB</strong> (25,894 bytes, 891 passengers) — about 1/40 of a megabyte.</p>
  </div>
  <div style="margin: 20px 0;">
    <div style="display: flex; flex-wrap: wrap; align-items: baseline; justify-content: space-between; gap: 4px 16px; margin-bottom: 8px;">
      <strong style="color: #2dd4bf; font-size: 18px;">1 GB</strong>
      <span style="color: #e2e8f0; font-size: 15px;">1 GB = 1,024 MB</span>
    </div>
    <div style="height: 18px; background: #243247; border-radius: 4px; overflow: hidden;">
      <div style="width: 50%; height: 100%; background: #2dd4bf; border-radius: 4px;"></div>
    </div>
    <p style="margin: 9px 0 0; color: #cbd5e1; font-size: 14px; line-height: 1.65;"><strong>A 4K movie:</strong> <em>Forrest Gump</em> runs <a href="https://www.paramount.com/movies/forrest-gump" style="color: #7dd3fc; text-decoration: underline;" target="_blank" rel="noopener">142 minutes</a>. At Netflix’s <a href="https://help.netflix.com/en/node/87" style="color: #7dd3fc; text-decoration: underline;" target="_blank" rel="noopener">up to 7 GB per hour for 4K</a>, a movie of that length would use <strong>up to about 16.6 GB</strong> of streaming data (142 ÷ 60 × 7). This is an estimate; a downloaded file’s size depends on compression.</p>
  </div>
  <div style="margin: 20px 0;">
    <div style="display: flex; flex-wrap: wrap; align-items: baseline; justify-content: space-between; gap: 4px 16px; margin-bottom: 8px;">
      <strong style="color: #fbbf24; font-size: 18px;">1 TB</strong>
      <span style="color: #e2e8f0; font-size: 15px;">1 TB = 1,024 GB</span>
    </div>
    <div style="height: 18px; background: #243247; border-radius: 4px; overflow: hidden;">
      <div style="width: 75%; height: 100%; background: #fbbf24; border-radius: 4px;"></div>
    </div>
    <p style="margin: 9px 0 0; color: #cbd5e1; font-size: 14px; line-height: 1.65;"><strong>A personal computer:</strong> a <a href="https://www.seagate.com/gb/en/support/internal-hard-drives/desktop-hard-drives/desktop-hdd/" style="color: #7dd3fc; text-decoration: underline;" target="_blank" rel="noopener">1–2 TB desktop hard drive</a> can hold your documents, photos, games, and many movies.</p>
  </div>
  <div style="margin: 20px 0;">
    <div style="display: flex; flex-wrap: wrap; align-items: baseline; justify-content: space-between; gap: 4px 16px; margin-bottom: 8px;">
      <strong style="color: #fb7185; font-size: 18px;">1 PB</strong>
      <span style="color: #e2e8f0; font-size: 15px;">1 PB = 1,024 TB</span>
    </div>
    <div style="height: 18px; background: #243247; border-radius: 4px; overflow: hidden;">
      <div style="width: 100%; height: 100%; background: #fb7185; border-radius: 4px;"></div>
    </div>
    <p style="margin: 9px 0 0; color: #cbd5e1; font-size: 14px; line-height: 1.65;"><strong>Industry data:</strong> <a href="https://aws.amazon.com/blogs/aws/aws-services-scale-to-new-heights-for-prime-day-2025-key-metrics-and-milestones/" style="color: #7dd3fc; text-decoration: underline;" target="_blank" rel="noopener">Amazon Aurora, Prime Day 2025</a> stored <strong>4,071 TB (about 4 PB)</strong> across its databases. The <a href="https://netflixtechblog.com/optimizing-data-warehouse-storage-7b94a48fdcbe" style="color: #7dd3fc; text-decoration: underline;" target="_blank" rel="noopener">Netflix data warehouse, reported in 2020</a> held <strong>hundreds of PB</strong>. These figures describe collections of databases or warehouse storage.</p>
  </div>
  <div style="margin-top: 18px; padding-top: 12px; border-top: 1px solid #334155; color: #cbd5e1; font-size: 13px; line-height: 1.65;">
    <strong style="color: #f1f5f9;">Reading the bars:</strong> this is a logarithmic step diagram. Each equal increase in length represents 1,024× more data; the bar lengths are not linear byte counts.<br>
    <strong style="color: #f1f5f9;">Unit note:</strong> the 1,024-step classroom convention above uses MB, GB, TB, PB as shorthand for the formal binary units MiB, GiB, TiB, PiB. Standard decimal MB, GB, TB, PB use steps of 1,000. The real-world examples retain their sources’ reported units. See <a href="https://physics.nist.gov/cuu/Units/binary.html" style="color: #7dd3fc; text-decoration: underline;" target="_blank" rel="noopener">NIST unit definitions</a>.
  </div>
</div>

**Why this matters:** under the 1,024-step convention, 1 PB is **1,024 × 1,024 × 1,024 = 1,073,741,824 times** the size of 1 MB. A pandas DataFrame usually lives in memory; a database can keep the full table on disk and return just the rows and columns your query needs.


<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/sql_1.jpg" width="800">
</p>

Figure source: Generated by ChatGPT

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/sql_2.png" width="500">
</p>

Figure source: [Linkedin Post](https://www.linkedin.com/pulse/datasets-vs-databases-sarah-mabrouk-kfbcf/)

## 2.8 Create a database, a table, and some rows
Three steps: **connect** (this creates the file if it does not exist), get a **cursor** to run statements, and **commit** to save your changes.

We will start by hand-building a small table so you can see the machinery, then take the fast route with pandas.

In [ ]:
import sqlite3

# 1. create a new empty database
#  connect -- creates course.db in this Colab session if it is not there yet
conn = sqlite3.connect("course.db")

# 2. iniate a cur object, which carries your statement to the `course.db`
cur = conn.cursor()

# 3. drop the table `holdings` if we already created it
cur.execute("DROP TABLE IF EXISTS holdings")

# 4. create a table by declaring its columns and their types
# create new columns: ["ticker", "price", "chg"].
# TEXT means strings, and REAL means decimals/floats
cur.execute("CREATE TABLE holdings (ticker TEXT, price REAL, chg REAL)") # this creation is pending

# 5. approve the task above
conn.commit()
print("table created ✅")

To insert rows, hand `executemany` a **list of tuples**. The `?` marks are placeholders that SQLite fills in safely — never paste values into a query string by hand.

In [ ]:
# 6. Suppose we have table: Note this is a list of tupple
stocks = [
    ("NVDA", 219.00, -3.42),
    ("AAPL", 146.00, -2.73),
    ("GOOG", 2829.27, -58.20),
]

# 7. Delete all contents in the iniated `holdings` table
# DROP TABLE in the previous cell is to delete the whole table.
cur.execute("DELETE FROM holdings")

# 8.1 write every row into the `holdings` table
cur.executemany("INSERT INTO holdings VALUES (?, ?, ?)", stocks)

# 8.2 this style also works
#cur.executemany("INSERT INTO holdings (ticker, price, chg) VALUES (?, ?, ?)", stocks)

# # 8.3 This is same to
# for row in stocks:
#     cur.execute("INSERT INTO holdings VALUES (?, ?, ?)", row)

# 9. confirm all new executions.
conn.commit()

print("rows inserted:", cur.rowcount)  # only count for the last execution

## 2.9 Querying with SQL: `SELECT`, `WHERE`, `ORDER BY`
A query answers: *which columns* (`SELECT`), *from which table* (`FROM`), *which rows* (`WHERE`), and *in what order* (`ORDER BY`).

The friendliest way to run one is **`pd.read_sql_query`**, which hands the result straight back as a **DataFrame** — so everything you learned in Session 03 still applies.

In [ ]:
# every column, every row
df_holdings = pd.read_sql_query("SELECT * FROM holdings", conn)
print(df_holdings)

A commonly used SQL query syntax:  
```sql
SELECT  col_name_1, col_name_2      
FROM    table_name
WHERE   conditions
ORDER BY col_name_1 DESC
```

In [ ]:
# pick columns, filter rows, and sort -- all in one statement
query = """
SELECT ticker, price
FROM holdings
WHERE price > 200
ORDER BY price DESC
"""
pd.read_sql_query(query, conn)

In [ ]:
query2 = """
SELECT ticker, price, chg
FROM holdings
WHERE price > 200 AND chg < -3
ORDER BY price
"""

pd.read_sql_query(query2, conn)

If you want plain Python objects instead of a DataFrame, run the query on the cursor and call **`.fetchall()`**, which returns a list of tuples.

In [ ]:
rows = cur.execute("SELECT ticker, price " \
                    "FROM holdings " \
                    "WHERE price > 200").fetchall()  # fetch all means: extract all
print(rows)
print("first ticker:", rows[0][0])   # a list of tuples, indexed as usual

## 2.10 Aggregating with `GROUP BY`
SQL has the same summary functions you have been using: `COUNT`, `AVG`, `MIN`, `MAX`, `SUM`. Put them with `GROUP BY` and you have SQL's version of `groupby().agg()`.

Let us switch to the real data. **`to_sql`** writes a whole DataFrame into the database in one line — this is the bridge between the two halves of today's session.

In [ ]:
titanic.head()

In [ ]:
# write the two tables into the course.db database
# (1) write the titanic table and give it a new name: passengers
titanic.to_sql("passengers",        # new name in the database
               conn,                # the active db
               index=False,         # drop the index col, since we already have id column
               if_exists="replace")

# (2) write the ports table
ports.to_sql("ports",
             conn,
             index=False,
             if_exists="replace")

conn.commit()

# which tables now live in this database?
print(pd.read_sql_query("SELECT name " \
                        "FROM sqlite_master " \
                        "WHERE type='table'", conn))

In [ ]:
# survival rate and average fare per class -- compare this with the pandas version above
query = """
SELECT pclass,
       COUNT(*)                AS n_passengers,
       ROUND(AVG(survived), 3) AS survival_rate,
       ROUND(AVG(fare), 2)     AS avg_fare
FROM passengers
GROUP BY pclass
ORDER BY pclass
"""
pd.read_sql_query(query, conn)

Line that up against `titanic.groupby("pclass")["survived"].mean()` from §2.5 — identical numbers, different accent. `AS` just renames a column in the output, the way `rename` does in pandas.

> ⚠️ **`COUNT(*)` and pandas' `.count()` are not the same thing.** `COUNT(*)` counts **rows**, including rows where a column is missing. pandas' `.count()` counts **non-missing values**. On a column with no gaps they agree; on a column with gaps they will not, and the difference is exactly the number of missing values. Neither is wrong — but if you compare a SQL summary against a pandas one, that is the first place the numbers diverge.

In [ ]:
# the same question, two ways -- watch the counts for the age column
sql_count = pd.read_sql_query("SELECT COUNT(*) AS n_rows, COUNT(age) AS n_ages FROM passengers", conn)
print(sql_count)
print("pandas .count() on age:", titanic["age"].count(), "| rows:", titanic.shape[0])
# COUNT(*) = 891 rows, but only 714 of them actually record an age.

## 2.11 `JOIN` — the SQL cousin of `merge`
Remember gluing the port names onto the passenger list with `pd.merge`? In SQL that is a **`JOIN`**, and the `ON` clause names the shared key. A `LEFT JOIN` keeps every row of the left table, exactly like `how="left"`.

In [ ]:
query = """
SELECT p.passenger_id, p.sex, p.fare, o.port_name, o.country
FROM passengers AS p
LEFT JOIN ports AS o
  ON p.embarked = o.embarked
LIMIT 5
"""
pd.read_sql_query(query, conn)

`p` and `o` are short nicknames (aliases) for the two tables, so `p.fare` reads "the `fare` column of `passengers`". `LIMIT 5` returns just the first five rows — SQL's `.head()`.

Now a join *and* an aggregate together, which is where SQL starts to feel powerful:

In [ ]:
query = """
SELECT o.port_name,
       COUNT(*)                AS n_passengers,
       ROUND(AVG(p.fare), 2)   AS avg_fare,
       ROUND(AVG(p.survived), 3) AS survival_rate
FROM passengers AS p
LEFT JOIN ports AS o
  ON p.embarked = o.embarked
GROUP BY o.port_name
ORDER BY n_passengers DESC
"""
pd.read_sql_query(query, conn)

When you are finished with a database, **close** the connection. (In a notebook you often leave it open while you work, but closing is the tidy habit.)

In [ ]:
conn.commit()
conn.close()
print("connection closed ✅")
print("the database file is still on disk:", os.path.exists("course.db"))

### A streaming home screen is a stack of queries

Once you know what to look for, a streaming home screen starts looking like `SELECT` statements. **Run the cell below, then click any strip** to see its SQL and explanation: profiles, Continue Watching, Top 10, recommendations, or saving your place when you pause.

These are simplified teaching examples adapted from `why_sql_not_excel.html`.


In [ ]:
#@title Interactive streaming demo { display-mode: "form" }
from IPython.display import display
from html import escape

# Adapted from the home-screen section of why_sql_not_excel.html.
# All HTML, CSS, and JavaScript are embedded so this also works in Colab.
streaming_home_html = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>A streaming home screen is a stack of queries</title>
<style>
:root {
  --ground:      #0D1117;
  --surface:     #151B24;
  --surface-2:   #1C2430;
  --surface-3:   #232D3B;
  --rule:        #2C3846;
  --rule-soft:   #212B36;

  --ink:         #E8ECF1;
  --ink-mid:     #AEB9C7;
  --ink-soft:    #7E8B9B;

  --amber:       #E9A63E;   /* scale, strain, the spreadsheet side */
  --amber-dim:   #8A6220;
  --teal:        #52C0A4;   /* the database side */
  --teal-dim:    #2C6D5D;
  --brick:       #E4705C;   /* failure */
  --brick-dim:   #7A3529;
  --on-accent:   #10141A;   /* text that sits on top of --amber */

  --shadow: 0 1px 2px rgba(0,0,0,.5), 0 12px 32px -12px rgba(0,0,0,.7);
  --maxw: 74ch;

  --font-display: "Bahnschrift", "DIN Alternate", "Avenir Next Condensed",
                  "Helvetica Neue", "Segoe UI", system-ui, sans-serif;
  --font-body:    "Charter", "Bitstream Charter", "Constantia", "Sitka Text",
                  "Iowan Old Style", Georgia, "Times New Roman", serif;
  --font-mono:    "Cascadia Mono", "SF Mono", Menlo, Consolas, ui-monospace, monospace;

  color-scheme: dark;
}
.panel {
  background: var(--surface); border: 1px solid var(--rule); border-radius: 4px;
  box-shadow: var(--shadow); overflow: hidden;
}
.panel-head {
  display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: .8rem;
  padding: .85rem 1.15rem; border-bottom: 1px solid var(--rule); background: var(--surface-2);
}
.panel-head .title {
  font-family: var(--font-display); font-variation-settings: 'wght' 600, 'wdth' 86;
  font-size: .82rem; letter-spacing: .14em; text-transform: uppercase; color: var(--ink-mid);
}
.panel-body { padding: clamp(1.1rem, 2.4vw, 1.8rem); }

.grid-2 { display: grid; gap: 1.15rem; grid-template-columns: repeat(auto-fit, minmax(275px, 1fr)); }
.grid-3 { display: grid; gap: 1.15rem; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); }


.sql {
  font-family: var(--font-mono); font-size: .84rem; line-height: 1.75;
  background: var(--surface-2); border: 1px solid var(--rule); border-left: 3px solid var(--teal);
  border-radius: 3px; padding: .9rem 1.1rem; margin: 0;
  white-space: pre; overflow-x: auto; color: var(--ink);
}
.sql .kw  { color: var(--amber); font-weight: 700; }
.sql .str { color: var(--teal); }
.sql .cm  { color: var(--ink-soft); font-style: italic; }
.sql.off  { border-left-color: var(--rule); }


.screen {
  --amber: #E9A63E;
  --ink-soft: #7D8B9C;
  background: #0B0E13; border: 1px solid var(--rule); border-radius: 4px; padding: 1rem;
  display: flex; flex-direction: column; gap: 1.1rem;
}
.screen .srow { display: flex; flex-direction: column; gap: .45rem; }
.screen .srow > b {
  font-family: var(--font-display); font-variation-settings: 'wght' 600, 'wdth' 90;
  font-size: .8rem; letter-spacing: .06em; color: #C9D2DE; font-weight: 600;
}
.tiles { display: flex; gap: .45rem; overflow-x: auto; padding-bottom: .2rem; }
.tile {
  flex: 0 0 84px; height: 52px; border-radius: 3px; border: 1px solid #222C39;
  background: linear-gradient(150deg, #1B2430, #131A23);
  display: grid; place-items: center; font-family: var(--font-mono); font-size: .58rem;
  color: #7D8B9C; text-align: center; padding: .2rem; line-height: 1.2;
}
.probe {
  all: unset; cursor: pointer; display: block; border-radius: 4px; padding: .5rem;
  border: 1px dashed transparent; transition: border-color .15s, background .15s;
}
.probe:hover, .probe[aria-pressed="true"] { border-color: var(--amber); background: rgba(233,166,62,.06); }
.probe[aria-pressed="true"] { border-style: solid; }


* { box-sizing: border-box; }
html { color-scheme: dark; }
body { margin: 0; padding: 20px; background: var(--ground); color: var(--ink);
       font-family: system-ui, sans-serif; font-size: 15px; line-height: 1.6; }
p { margin: 0; }
code { font-family: var(--font-mono); font-size: .9em; }
.stack { display: flex; flex-direction: column; gap: 1rem; }
.stack-lg { display: flex; flex-direction: column; gap: 1.5rem; }
.col { max-width: 90ch; }
.grid-2 { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1.2fr); gap: 20px; }
.grid-2 > * { min-width: 0; }
.probe { min-width: 0; max-width: 100%; box-sizing: border-box; }
.srow, .tiles { min-width: 0; max-width: 100%; }
.sql { white-space: pre-wrap; overflow-wrap: anywhere; font-size: 12px; line-height: 1.8; }
.panel-body { padding: 18px; }
.screen .srow > b { font-size: 13px; letter-spacing: .02em; }
.tile { font-size: 11px; }
.probe:focus-visible { outline: 2px solid var(--amber); outline-offset: 3px; }
@media (max-width: 720px) { .grid-2 { grid-template-columns: minmax(0, 1fr); } body { padding: 14px; } }
@media (prefers-reduced-motion: reduce) { .probe { transition: none; } }
</style>
</head>
<body>
<section id="s4">
  <div class="stack-lg">
<div class="grid-2" style="align-items:start">
      <div class="screen" role="group" aria-label="Sketch of a streaming home screen">
        <button type="button" class="probe" data-q="profiles" aria-pressed="false">
          <div class="srow"><b>Who's watching?</b>
            <div class="tiles"><span class="tile">Ana</span><span class="tile">Ben</span><span class="tile">Kids</span></div>
          </div>
        </button>
        <button type="button" class="probe" data-q="continue" aria-pressed="true">
          <div class="srow"><b>Continue Watching for Ana</b>
            <div class="tiles"><span class="tile">S4·E2<br>00:23:41</span><span class="tile">Ep 7<br>00:41:05</span><span class="tile">00:12:58</span></div>
          </div>
        </button>
        <button type="button" class="probe" data-q="top10" aria-pressed="false">
          <div class="srow"><b>Top 10 in the U.S. Today</b>
            <div class="tiles"><span class="tile">1</span><span class="tile">2</span><span class="tile">3</span><span class="tile">4</span><span class="tile">5</span></div>
          </div>
        </button>
        <button type="button" class="probe" data-q="because" aria-pressed="false">
          <div class="srow"><b>Because you watched a documentary</b>
            <div class="tiles"><span class="tile">doc</span><span class="tile">doc</span><span class="tile">doc</span><span class="tile">doc</span></div>
          </div>
        </button>
        <button type="button" class="probe" data-q="resume" aria-pressed="false">
          <div class="srow"><b>▸ You press pause at 00:23:41</b>
            <div class="tiles"><span class="tile" style="flex-basis:180px">one row changes</span></div>
          </div>
        </button>
      </div>

      <div class="panel">
        <div class="panel-head"><span class="title" id="qTitle">Continue Watching</span></div>
        <div class="panel-body stack">
          <pre class="sql" id="qSql" aria-live="polite"></pre>
          <p id="qEn" style="font-size:.92rem;color:var(--ink-mid)"></p>
        </div>
      </div>
    </div>

    <div class="col stack">
      <p>
        Notice what every one of those has in common: <em>pick some columns, keep some rows, put them in
        an order, stop after a few.</em> That is not a large vocabulary. Six keywords cover the vast
        majority of SQL anyone writes, and you have been doing all six in <code>pandas</code> earlier in this notebook
        without calling them by these names.
      </p>
    </div>
  </div>
</section>
<script>
(function () {
  const Q = {
    profiles: {
      t: "Who's watching?",
      sql: "<span class='kw'>SELECT</span> profile_id, display_name, avatar_url\n" +
           "<span class='kw'>FROM</span>   profiles\n" +
           "<span class='kw'>WHERE</span>  account_id = <span class='str'>8814207</span>\n" +
           "<span class='kw'>ORDER BY</span> created_at;",
      en: "One account, several people. The profiles table stores a row per person and a foreign key " +
          "back to the account that pays. This is exactly the pd.merge() relationship from Session 04, " +
          "written the other way round."
    },
    continue: {
      t: "Continue Watching",
      sql: "<span class='kw'>SELECT</span>   t.title, w.episode, w.position_seconds\n" +
           "<span class='kw'>FROM</span>     watch_progress w\n" +
           "<span class='kw'>JOIN</span>     titles t <span class='kw'>ON</span> t.title_id = w.title_id\n" +
           "<span class='kw'>WHERE</span>    w.profile_id = <span class='str'>42</span>\n" +
           "  <span class='kw'>AND</span>    w.finished  = <span class='str'>0</span>\n" +
           "<span class='kw'>ORDER BY</span> w.last_watched_at <span class='kw'>DESC</span>\n" +
           "<span class='kw'>LIMIT</span>    10;",
      en: "Six lines. Note LIMIT 10 — the row on screen holds ten tiles, so the database is asked for " +
          "ten rows, not for the table. It never touches the other 325 million members' progress."
    },
    top10: {
      t: "Top 10 in the U.S. Today",
      sql: "<span class='kw'>SELECT</span>   t.title, <span class='kw'>COUNT</span>(*) <span class='kw'>AS</span> plays\n" +
           "<span class='kw'>FROM</span>     plays p\n" +
           "<span class='kw'>JOIN</span>     titles t <span class='kw'>ON</span> t.title_id = p.title_id\n" +
           "<span class='kw'>WHERE</span>    p.country = <span class='str'>'US'</span>\n" +
           "  <span class='kw'>AND</span>    p.played_at &gt;= <span class='kw'>CURRENT_DATE</span>\n" +
           "<span class='kw'>GROUP BY</span> t.title\n" +
           "<span class='kw'>ORDER BY</span> plays <span class='kw'>DESC</span>\n" +
           "<span class='kw'>LIMIT</span>    10;",
      en: "A GROUP BY over hundreds of millions of play events, reduced to ten rows. You wrote the " +
          "identical thing as titanic.groupby('pclass')['survived'].mean() — same operation, " +
          "seven orders of magnitude more data."
    },
    because: {
      t: "Because you watched…",
      sql: "<span class='kw'>SELECT</span>   t.title\n" +
           "<span class='kw'>FROM</span>     titles t\n" +
           "<span class='kw'>JOIN</span>     title_genres g <span class='kw'>ON</span> g.title_id = t.title_id\n" +
           "<span class='kw'>WHERE</span>    g.genre = <span class='str'>'Documentary'</span>\n" +
           "  <span class='kw'>AND</span>    t.title_id <span class='kw'>NOT IN</span> (\n" +
           "        <span class='kw'>SELECT</span> title_id <span class='kw'>FROM</span> watch_progress\n" +
           "        <span class='kw'>WHERE</span>  profile_id = <span class='str'>42</span>)\n" +
           "<span class='kw'>ORDER BY</span> t.popularity <span class='kw'>DESC</span>\n" +
           "<span class='kw'>LIMIT</span>    20;",
      en: "The real recommender is a machine-learning model, but the model's inputs and its output " +
          "shortlist both arrive by query. 'Things like this that you have not already seen' is a " +
          "JOIN plus a NOT IN — and NOT IN is why the row never shows you last night's episode."
    },
    resume: {
      t: "Pressing pause",
      sql: "<span class='kw'>UPDATE</span> watch_progress\n" +
           "<span class='kw'>SET</span>    position_seconds = <span class='str'>1421</span>,\n" +
           "       last_watched_at  = <span class='kw'>CURRENT_TIMESTAMP</span>\n" +
           "<span class='kw'>WHERE</span>  profile_id = <span class='str'>42</span>\n" +
           "  <span class='kw'>AND</span>  title_id   = <span class='str'>70143836</span>;\n\n" +
           "<span class='cm'>-- exactly one row is touched. the rest of the table never moves.</span>",
      en: "This is the one that kills the CSV argument outright. In a file, changing one number in the " +
          "middle means rewriting the whole file, because the rows after it may shift. A database " +
          "changes that row's page on disk and nothing else — which is why your phone knows where you " +
          "stopped before you have finished picking it up."
    }
  };

  const probes = document.querySelectorAll(".probe");
  const tEl = document.getElementById("qTitle");
  const sEl = document.getElementById("qSql");
  const eEl = document.getElementById("qEn");

  function show(key, btn) {
    probes.forEach(p => p.setAttribute("aria-pressed", String(p === btn)));
    tEl.textContent = Q[key].t;
    sEl.innerHTML   = Q[key].sql;
    eEl.textContent = Q[key].en;
  }
  probes.forEach(p => p.addEventListener("click", () => show(p.dataset.q, p)));
  show("continue", document.querySelector('.probe[data-q="continue"]'));
})();
</script>
</body>
</html>"""

# srcdoc keeps the widget self-contained and isolates its styles from the notebook.
streaming_home_iframe = (
    '<iframe title="A streaming home screen is a stack of queries" '
    'data-lecture-demo="streaming-home" sandbox="allow-scripts" '
    'style="width:100%;height:780px;border:0;border-radius:10px;" '
    'srcdoc="' + escape(streaming_home_html, quote=True) + '"></iframe>'
)

# Hide only this demo's code input in Jupyter Notebook and JupyterLab.
# Colab uses the display-mode form annotation at the top of this cell.
streaming_home_input_style = """<style>
.jp-CodeCell:has(iframe[data-lecture-demo="streaming-home"]) > .jp-Cell-inputWrapper,
.code_cell:has(iframe[data-lecture-demo="streaming-home"]) > .input {
    display: none !important;
}
</style>"""
display({"text/html": streaming_home_input_style + streaming_home_iframe}, raw=True)


#### ✏️ Try It Yourself — Exercise 5
**Difficulty:** ★★★★★  **Skills:** `sqlite3.connect`, `to_sql`, `pd.read_sql_query`, `WHERE`, `GROUP BY`, `ORDER BY`

Re-open a connection to `course.db`, then answer these with **SQL** (each one query, read back with `pd.read_sql_query`):
(a) the `passenger_id`, `age`, and `fare` of the 5 passengers who paid the **highest fares**;
(b) the number of passengers and the **average age** for each `sex`;
(c) how many passengers in class 3 **survived** (`survived = 1`).

Close the connection when you are done.

In [ ]:
# Your code here
conn = sqlite3.connect("course.db")

print(pd.read_sql_query("SELECT name " \
                        "FROM sqlite_master " \
                        "WHERE type='table'", conn))
print(pd.read_sql_query("SELECT * FROM passengers LIMIT 5", conn))

In [ ]:
# (a)
query = """
SELECT passenger_id, age, fare
FROM passengers
ORDER BY fare DESC
LIMIT 5
"""

print(pd.read_sql_query(query, conn))


#(b)
query = """
SELECT  COUNT(*)            AS n_passenger,
        ROUND(AVG(age),2)   AS avg_age
FROM passengers
GROUP BY SEX
"""
print(pd.read_sql_query(query, conn))


# (c) how many passengers in class 3 **survived** (`survived = 1`).
query = """
SELECT  COUNT(*)        AS n_passenger,
        SUM(survived)   AS n_survival
FROM    passengers
WHERE   pclass = 3
"""

print(pd.read_sql_query(query, conn))


<details>
<summary>▶ Show solution</summary>

```python
conn = sqlite3.connect("course.db")

# (a) top 5 fares
q_a = """
SELECT passenger_id, age, fare
FROM passengers
ORDER BY fare DESC
LIMIT 5
"""
print(pd.read_sql_query(q_a, conn))

# (b) count and average age by sex
q_b = """
SELECT sex, COUNT(*) AS n, ROUND(AVG(age), 2) AS avg_age
FROM passengers
GROUP BY sex
"""
print(pd.read_sql_query(q_b, conn))

# (c) survivors in third class
q_c = """
SELECT COUNT(*) AS n_survivors
FROM passengers
WHERE pclass = 3 AND survived = 1
"""
print(pd.read_sql_query(q_c, conn))

conn.close()
```

</details>

## 2.12 pandas ↔ SQL, side by side
Keep this table somewhere you can find it. Once you see that the two languages express the same handful of ideas, picking up either one gets much easier.

| Task | pandas | SQL |
|---|---|---|
| all rows & columns | `df` | `SELECT * FROM t` |
| pick columns | `df[["a", "b"]]` | `SELECT a, b FROM t` |
| filter rows | `df[df["x"] > 5]` | `SELECT * FROM t WHERE x > 5` |
| two conditions | `df[(df.x > 5) & (df.y == 1)]` | `... WHERE x > 5 AND y = 1` |
| sort | `df.sort_values("x", ascending=False)` | `... ORDER BY x DESC` |
| first rows | `df.head(5)` | `... LIMIT 5` |
| count rows | `df.shape[0]` | `SELECT COUNT(*) FROM t` |
| group summary | `df.groupby("g")["x"].mean()` | `SELECT g, AVG(x) FROM t GROUP BY g` |
| join tables | `pd.merge(a, b, on="k", how="left")` | `FROM a LEFT JOIN b ON a.k = b.k` |
| rename output | `df.rename(columns={"x": "y"})` | `SELECT x AS y FROM t` |

---
## 3. Conclusions & Key Takeaways
- `.describe()` and **`.agg([...])`** build descriptive tables; when a column is **skewed**, report the **median** alongside the mean.
- Tidy with `.rename()`, `.drop(columns=...)`, `.astype()`, and `.drop_duplicates()` — each returns a **new** table.
- Find gaps with `.isna().sum()`, then **choose** deliberately: `.dropna()` loses rows, `.fillna()` invents values, doing nothing is sometimes right. Say which you did.
- **`groupby` + `.agg()`** gives several statistics per group; **`pivot_table`** lays two grouping variables out as a readable grid.
- **`concat`** stacks rows; **`merge`** joins on a key. Always check the row count after a join.
- **SQLite** is a whole database in one file, built into Python. Write tables with **`to_sql`**, read queries back with **`pd.read_sql_query`**.
- SQL's `SELECT` / `WHERE` / `GROUP BY` / `JOIN` are the same ideas as pandas' column selection, masks, `groupby`, and `merge`. Watch the one gap: `COUNT(*)` counts rows, pandas' `.count()` counts non-missing values.

> 🐍 **A gotcha for later:** statistics computed by pandas or NumPy come back as `numpy` numbers (`int64`, `float64`), and `json.dump` refuses to write those. Wrap them in `int(...)` or `float(...)` first — e.g. `int(df.shape[0])`, `float(df["x"].mean())` — and JSON will take them happily.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| custom summary | `df["x"].agg(["mean", "median", "std"])` |
| rename / drop | `df.rename(columns={...})`, `df.drop(columns=[...])` |
| duplicates | `df.duplicated().sum()`, `df.drop_duplicates()` |
| missing values | `df.isna().sum()`, `df.dropna(subset=["x"])`, `df["x"].fillna(v)` |
| group + stats | `df.groupby("g")["x"].agg(["mean", "count"])` |
| summary → dict | `df.groupby("g")["x"].mean().to_dict()` |
| correlation (pair) | `df["a"].corr(df["b"])` |
| pivot grid | `pd.pivot_table(df, values="v", index="r", columns="c", aggfunc="mean")` |
| stack / join | `pd.concat([a, b])`, `pd.merge(a, b, on="k", how="left")` |
| open a database | `conn = sqlite3.connect("my.db")` |
| DataFrame → table | `df.to_sql("t", conn, index=False, if_exists="replace")` |
| query → DataFrame | `pd.read_sql_query("SELECT * FROM t", conn)` |
| insert rows | `cur.executemany("INSERT INTO t VALUES (?, ?)", rows)` |
| save / close | `conn.commit()`, `conn.close()` |

### ⏭️ Coming Up Next
**Control structures and iterables.** At last we meet `for` and `if` — the tools we have deliberately done without. You will find that pandas already taught you *why* vectorized thinking is usually better, which makes loops much easier to use well. Read McKinney Ch 3.

### 📌 This Week's Assignment
A pandas + SQLite assignment on the **penguins** dataset — see the `assignment/` folder / GitHub.

> 💡 **Tip:** to learn SQL quickly, write the same question twice — once with pandas, once as a query — and check that the two answers agree. Do that three or four times and it will stick.